# 08 - MCP Server Integration

> **When to use**: When you want AI assistants (Claude Desktop, Cursor) to directly operate the database to generate test data.
>
> **Core concept**: MCP (Model Context Protocol) lets AI assistants directly invoke sqlseed via two offline tools.

## Applicable Scenarios

- Use natural language to let AI generate test data → MCP Server
- AI assistant needs to generate config → `sqlseed_generate_yaml`
- AI assistant needs to execute fill → `sqlseed_execute_fill`

## What You Will Learn

- MCP Server installation and configuration
- Rule-driven YAML generation and data filling without an LLM
- Security validation mechanism
- Separate AI MCP server for optional model-backed features

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| **→ 08** | **MCP Server Integration** | **Plugins: MCP** | **01** |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [ ]:
from __future__ import annotations

# Run from examples/notebooks. Install from the repository root in one resolution:
# python -m pip install -e ".[dev,all]" -e "./plugins/sqlseed-cli" \
#   -e "./plugins/sqlseed-ai[dev,mcp]" -e "./plugins/mcp-server-sqlseed" -e "./plugins/sqlseed-web[dev]"
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

import sqlseed
from sqlseed import connect

sys.path.insert(0, str(Path("..").resolve()))  # build_demo_db only
from build_demo_db import build

# Keep this object alive across cells. No existing database is opened or rebuilt.
_demo_directory = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-")
demo_root = Path(_demo_directory.name)
os.environ["SQLSEED_CACHE_DIR"] = str(demo_root / "cache")
db_path = build(demo_root / "demo.db")


def require(condition, message):
    """Stop the tutorial if an expected outcome did not occur."""
    if not condition:
        raise RuntimeError(message)


def check_generation(result, count):
    """Check errors and generated row count before showing success."""
    require(not result.errors and result.count == count, f"Generation failed: {result.errors}; count={result.count}")


def read_rows(database, sql):
    """Read actual persisted values using a fixed tutorial query."""
    # The queries below are fixed tutorial SQL, never external identifiers.
    with closing(sqlite3.connect(database)) as connection:
        return connection.execute(sql).fetchall()


with connect(str(db_path), provider="faker") as orch:
    for table, count in (("organizations", 5), ("members", 20), ("projects", 10), ("tags", 8)):
        check_generation(orch.fill_table(table, count=count, seed=42, skip_ai=True), count)

print(f"sqlseed {sqlseed.__version__} | Temporary database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Offline MCP Tools | `plugins/mcp-server-sqlseed/src/mcp_server_sqlseed/server.py` | `mcp` |

> Corresponding architecture diagram: [§10 MCP Server Architecture](../../docs/architecture.zh-CN.md#10-mcp-服务器架构)

## 1. Rule-Driven YAML → Fill

The `mcp-server-sqlseed` package exposes `sqlseed_generate_yaml` and `sqlseed_execute_fill`. They use Core and require no API key. The executable speaks MCP to an assistant; this notebook calls the same tool functions directly to inspect their real results.

For model-backed tools, install `sqlseed-ai[mcp]` and configure its separate `mcp-server-sqlseed-ai` executable. This notebook does not test live LLM inference.


## 2. What the Offline Server Provides

- `sqlseed_generate_yaml(db_path, table_name)` maps schema columns to a reviewable YAML template.
- `sqlseed_execute_fill(db_path, table_name, count, yaml_config, enrich)` fills the selected table.

There is no schema-inspection tool or schema resource URI in this server. The Python API can inspect schema directly; use a separate database MCP if an assistant needs additional database tools.


## 3. Claude Desktop / Cursor Configuration

Add the sqlseed server to the AI assistant's MCP config:

In [ ]:
import json
import sysconfig

executable = Path(sysconfig.get_path("scripts")) / (
    "mcp-server-sqlseed.exe" if os.name == "nt" else "mcp-server-sqlseed"
)
config = {"mcpServers": {"sqlseed": {"command": str(executable)}}}
print(json.dumps(config, indent=2))
print("Use the executable from the environment containing the installed MCP package. No API key is needed.")

## 4. Inspect Schema with the Python API

This local context inspection is not a third MCP tool.


In [ ]:
with connect(str(db_path), provider="faker") as orch:
    context = orch.get_schema_context("tags")
require(bool(context["columns"]), "Expected real column metadata")
print(json.dumps(context, indent=2, ensure_ascii=False, default=str)[:800])

## 5. Tool 1: sqlseed_generate_yaml

Generate an offline template, parse it, and review the target table. No `api_key` argument exists on this tool.


In [ ]:
import yaml
from mcp_server_sqlseed.server import sqlseed_generate_yaml

yaml_config = sqlseed_generate_yaml(str(db_path), table_name="tags")
document = yaml.safe_load(yaml_config)
require(isinstance(document, dict), f"YAML generation failed: {yaml_config}")
require(document["tables"][0]["name"] == "tags", "Wrong YAML target")
# This independent table has no existing dependent rows in the demo.
# A fixed seed and clear flag make this cell repeatable.
document["tables"][0].update(seed=42, clear_before=True)
yaml_config = yaml.safe_dump(document, sort_keys=False)
print(yaml_config)

## 6. Tool 2: sqlseed_execute_fill

Use the generated YAML. The tool arguments choose the database, table and count; YAML only supplies that table's columns, clear flag and seed. A tool error can be returned as an `error` key or a nonempty `errors` list, so check both before reporting success.


In [ ]:
from mcp_server_sqlseed.server import sqlseed_execute_fill

members_before = read_rows(db_path, "SELECT member_id FROM members ORDER BY member_id")
result = sqlseed_execute_fill(str(db_path), table_name="tags", count=3, yaml_config=yaml_config)
require("error" not in result and not result.get("errors"), f"MCP fill failed: {result}")
require(result["count"] == 3, f"Wrong MCP count: {result}")
require(len(read_rows(db_path, "SELECT tag_id FROM tags")) == 3, "MCP did not persist three tags")
require(
    read_rows(db_path, "SELECT member_id FROM members ORDER BY member_id") == members_before,
    "MCP modified unrelated members",
)
print(json.dumps(result, indent=2))

## 7. Rejected Inputs Leave Data Unchanged

File targets must exist and use a supported SQLite suffix; database URLs are delegated to the adapter. Tables must be present in the target database. This is input validation, not a filesystem sandbox or an authorization system: give the server only databases you intend the assistant to access.


In [ ]:
tags_before = read_rows(db_path, "SELECT tag_id, name FROM tags ORDER BY tag_id")
invalid_table = sqlseed_execute_fill(str(db_path), table_name="missing_table", count=1)
require("error" in invalid_table, "Unknown table should fail")
invalid_yaml = sqlseed_execute_fill(str(db_path), table_name="tags", count=1, yaml_config="[]")
require("error" in invalid_yaml, "Non-mapping YAML should fail")
missing_file = sqlseed_generate_yaml(str(demo_root / "missing.db"), table_name="tags")
require(missing_file.startswith("# Error:"), "Missing file should fail")
require(
    read_rows(db_path, "SELECT tag_id, name FROM tags ORDER BY tag_id") == tags_before, "Rejected input modified rows"
)
print("Unknown table, invalid YAML and missing file were rejected; existing data is unchanged.")

## Summary

| MCP tool | Function | LLM required |
|---|---|---|
| `sqlseed_generate_yaml` | Rule-driven configuration template | No |
| `sqlseed_execute_fill` | Fill the selected table | No |

AI MCP runs separately as `mcp-server-sqlseed-ai` from `sqlseed-ai[mcp]`. See the [AI guide](../../docs/gemma4-integration.md).

**Next**: [09-plugin-hooks.ipynb](09-plugin-hooks.ipynb) — Plugin System and Hook Lifecycle


In [ ]:
require(len(read_rows(db_path, "SELECT member_id FROM members")) == 20, "The examples changed unrelated members")
print("Tutorial operations completed with the original 20 demo members preserved.")